In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import re
import time

import torch
import torch.utils.tensorboard

import fishyrl as frl

# Load Model

In [ ]:
# Load config
cfg = frl.utilities.load_config('../examples/configs/RLGym.yaml', '../examples/configs/Dreamer_1m.yaml', '../examples/configs/Dreamer_Base.yaml')  # ()

# Checkpoint file if not starting from scratch
checkpoint_file = '../examples/runs/RLGym_Soccar_2026-04-23_03-22-26/checkpoint_0050001.pt'
start_environment_step = int(re.search(r'checkpoint_(\d+).pt', checkpoint_file).group(1)) if checkpoint_file is not None else 0

# Load environment and actions
envs = frl.dreamer.construct_envs(cfg=cfg)
actions = frl.dreamer.construct_actions(cfg=cfg)

# Construct models
world_model, actor_critic_model, utility_modules = frl.dreamer.construct_models(
    # env_obs_dim=math.prod(envs.obs_shape[1:]),  # TODO: Maybe readd default env params
    env_actions=actions,
    device='cuda',
    cfg=cfg,
)

# Load model and buffer
if checkpoint_file is not None:
    frl.dreamer.load_models(
        path=checkpoint_file,
        world_model=world_model,
        actor_critic_model=actor_critic_model,
        utility_modules=utility_modules,
    )

# Simulate

Make sure that you have the v0.8.2 binary (compatibile with `rlviser-py`) for `rlviser` (Linux/MacOS) or `rlviser.exe` (Windows) in the examples folder, as well as a folder `examples/collision_meshes`, extracted using [RLArenaCollitionDumper](https://github.com/ZealanL/RLArenaCollisionDumper).

In [18]:
# Create environment
envs = frl.environments.VectorizedRLGymEnvironment(allow_rendering=True)
obs, _ = envs.reset()
actions = posteriors = hidden_states = initializations = None

# Get device
device = world_model.parameters().__next__().device

# Loop until done
while not envs._ended[0]:
    # Record start time
    start_time = time.time()

    # Render
    envs.render()

    # Compute action
    ret = frl.dreamer.compute_actions(
        obs=torch.from_numpy(obs).to(device),
        world_model=world_model,
        actor_critic_model=actor_critic_model,
        actions=actions,
        posteriors=posteriors,
        hidden_states=hidden_states,
        initializations=initializations,
    )
    env_actions = ret['env_actions']
    actions = ret['actions']
    posteriors = ret['posterior']
    hidden_states = ret['hidden_state']

    # Step environment
    obs, reward, terminated, truncated, info = envs.step(env_actions)
    initializations = torch.from_numpy(terminated | truncated).to(dtype=bool, device=device)

    # Sleep for remaining time
    tick_duration = 1 / envs._envs[0].renderer.tick_rate
    to_wait = tick_duration - (time.time() - start_time)
    if to_wait < 0:
        print(f'Late computation by {10**3 * -to_wait:.2f} ms...')
    time.sleep(max(0, to_wait))



Late computation by 94.38 ms...
